In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np

def rgb2gray(rgb):
    return np.dot(rgb[...,:3], [0.2989, 0.5870, 0.1140])

img = rgb2gray(mpimg.imread("2025_GreenLight_Alignement/2025_11_25/GL_pos0.bmp"))[550:950,350:750]
img2 = rgb2gray(mpimg.imread("2025_GreenLight_Alignement/2025_10_29/GL_start.bmp"))[400:800,550:950]
imgplot = plt.imshow(img)
plt.show()
imgplot = plt.imshow(img2)
plt.show()

print(len(img))
print(len(img[1]))
#print(img2[1])

FileNotFoundError: [Errno 2] No such file or directory: '2025_GreenLight_Alignement/2025_11_25/GL_pos0.bmp'

In [ ]:
import scipy.optimize as opt
import statistics

def twoD_Gaussian(xy, amplitude, xo, yo, sigma_x, sigma_y, theta, offset):
    x, y = xy
    xo = float(xo)
    yo = float(yo)
    a = (np.cos(theta)**2)/(2*sigma_x**2) + (np.sin(theta)**2)/(2*sigma_y**2)
    b = -(np.sin(2*theta))/(4*sigma_x**2) + (np.sin(2*theta))/(4*sigma_y**2)
    c = (np.sin(theta)**2)/(2*sigma_x**2) + (np.cos(theta)**2)/(2*sigma_y**2)
    g = offset + amplitude*np.exp( - (a*((x-xo)**2) + 2*b*(x-xo)*(y-yo)
                            + c*((y-yo)**2)))
    return g.ravel()

# use your favorite image processing library to load an image
h, w = img.shape
data = img.ravel()
h2, w2 = img2.shape
data2 = img2.ravel()

x = np.linspace(0, w, w)
y = np.linspace(0, h, h)
x, y = np.meshgrid(x, y)
x2 = np.linspace(0, w2, w2)
y2 = np.linspace(0, h2, h2)
x2, y2 = np.meshgrid(x2, y2)

# initial guess of parameters
initial_guess = (1, 200, 200, 0.2, 0.2, 0, 50)
initial_guess2 = (1, 200, 200, 0.2, 0.2, 0, 50)

# find the optimal Gaussian parameters
popt, pcov = opt.curve_fit(twoD_Gaussian, (x, y), data, p0=initial_guess)
popt2, pcov2 = opt.curve_fit(twoD_Gaussian, (x2, y2), data2, p0=initial_guess2)

# create new data with these parameters
data_fitted = twoD_Gaussian((x, y), *popt)
data_fitted2 = twoD_Gaussian((x2, y2), *popt2)

print("done")

In [ ]:
# Elipse
from math import pi, cos, sin, sqrt, log

print(popt)

u=popt[1]       #x-position of the center
v=popt[2]       #y-position of the center
a=abs(popt[3]*2*sqrt(2 * log(2)))       #radius on the x-axis
b=abs(popt[4]*2*sqrt(2 * log(2)))       #radius on the y-axis
t_rot=-popt[5]   #rotation angle

t = np.linspace(0, 2*pi, 100)
Ell = np.array([a*np.cos(t) , b*np.sin(t)])
     #u,v removed to keep the same center location
R_rot = np.array([[cos(t_rot) , -sin(t_rot)],[sin(t_rot) , cos(t_rot)]])
     #2-D rotation matrix

Ell_rot = np.zeros((2,Ell.shape[1]))
for i in range(Ell.shape[1]):
    Ell_rot[:,i] = np.dot(R_rot,Ell[:,i])

plt.plot( u+Ell[0,:] , v+Ell[1,:] )     #initial ellipse
plt.plot( u+Ell_rot[0,:] , v+Ell_rot[1,:],'darkorange' )    #rotated ellipse
plt.grid(color='lightgray',linestyle='--')
plt.show()


######################################################### Image 2
print(popt2)

u2=popt2[1]       #x-position of the center
v2=popt2[2]       #y-position of the center
a2=abs(popt2[3]*2*sqrt(2 * log(2)))       #radius on the x-axis
b2=abs(popt2[4]*2*sqrt(2 * log(2)))       #radius on the y-axis
t_rot2=-popt2[5]   #rotation angle

t2 = np.linspace(0, 2*pi, 100)
Ell2 = np.array([a2*np.cos(t2) , b2*np.sin(t2)])
     #u,v removed to keep the same center location
R_rot2 = np.array([[cos(t_rot2) , -sin(t_rot2)],[sin(t_rot2) , cos(t_rot2)]])
     #2-D rotation matrix

Ell_rot2 = np.zeros((2,Ell2.shape[1]))
for i in range(Ell2.shape[1]):
    Ell_rot2[:,i] = np.dot(R_rot2,Ell2[:,i])

plt.plot( u2+Ell2[0,:] , v2+Ell2[1,:] )     #initial ellipse
plt.plot( u2+Ell_rot2[0,:] , v2+Ell_rot2[1,:],'darkorange' )    #rotated ellipse
plt.grid(color='lightgray',linestyle='--')
plt.show()

In [ ]:
print("Size 1 (pixel): ",round(a,2),round(b,2))
print("Size 2 (pixel): ",round(a2,2),round(b2,2))

pixel_to_mm = 0.0053 #pixel size 5.30 micrometers
area1 = abs(pi*b*pixel_to_mm*a*pixel_to_mm)
radius1 = sqrt(area1/pi)
print("Area 1 = ", area1, " mm^2")
print("Equivalent circle radius 1 = ", radius1, " mm")

area2 = pi*b2*pixel_to_mm*a2*pixel_to_mm
radius2 = sqrt(area2/pi)
print("Area 2 = ", area2, " mm^2")
print("Equivalent circle radius 2 = ", radius2, " mm")

#plt.imshow(data_fitted.reshape(h,w))
#plt.plot( u+Ell_rot[0,:] , v+Ell_rot[1,:],'darkorange' )    #rotated ellipse
#plt.show()
plt.imshow(img)
plt.plot( u+Ell_rot[0,:] , v+Ell_rot[1,:],'darkorange' )    #rotated ellipse
plt.show()

########################################### Image 2
#plt.imshow(data_fitted2.reshape(h2,w2))
#plt.plot( u2+Ell_rot2[0,:] , v2+Ell_rot2[1,:],'darkorange' )    #rotated ellipse
#plt.show()
plt.imshow(img2)
plt.plot( u2+Ell_rot2[0,:] , v2+Ell_rot2[1,:],'darkorange' )    #rotated ellipse
plt.show()

In [ ]:
###########################################
# Damage Thresholds
# DMLP900:
# 1.00 J/cm2  (532 nm, 10 Hz, 10 ns, Ø250 µm)
# 6.50 J/cm2  (1064 nm, 10 Hz, 12 ns, Ø250 µm)
###########################################
# Beam Parameters
pulseDuration = 10e-12 # s (-> 1ps)
# Average Power = 74 W (Considering 75 W max - 1 W for continuum)
# Repetition Rate = 100 kHz
# Diameter of circle with equivalent area to elipse = 2094 micrometers
###########################################
# Calcs in Gentec-Eo Peak Power Calculator:
peakPowerDensity = 4.298e+10 # W/cm2 = J/(s*cm2)
peakPowerDensityPulse = peakPowerDensity * pulseDuration
energyPulse = 0.65 #mJ

print(energyPulse/area*100)
print(peakPowerDensityPulse, " J/cm2")

In [ ]:
Pos 1 =  9.403012630169194  mm^2
Pos 2 =  6.4540740043707645  mm^2
Pos 3 =  4.32165222652745  mm^2
Pos 4 =  5.25655064851198  mm^2
Pos 5 =  3.910697548231252  mm^2
Pos 6 =  4.741484942247088  mm^2